# 02 — Model Validation
Verify that the LP constraints are binding correctly:
- Energy balance holds
- RPS constraint is satisfied (not violated, not over-satisfied)
- Peak adequacy is met
- All scenarios return Optimal status
- Shadow prices / dual variables analysis


In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from models.optimizer import NYEnergyOptimizer
from config import TECH_COLORS, CAPACITY_CREDIT, NY_BASELINE_CO2_MT, RENEWABLE_TECHS

opt = NYEnergyOptimizer()
print(f"Annual demand: {opt.annual_demand_twh:.2f} TWh")
print(f"Peak demand:   {opt.peak_demand_gw:.2f} GW")
print()
results = opt.run_all_scenarios()

## 1. Constraint Verification

In [ ]:
print("=" * 70)
print(f"{'Scenario':<22} {'Status':<10} {'Gen≥Demand':<12} {'RPS%':<8} {'Peak OK'}")
print("=" * 70)
for r in results:
    total_gen = sum(r.generation_twh.values())
    ren_gen = sum(r.generation_twh[t] for t in r.generation_twh if t in RENEWABLE_TECHS)
    actual_rps = ren_gen / total_gen if total_gen > 0 else 0

    from config import RESERVE_MARGIN, CAPACITY_CREDIT
    rm = RESERVE_MARGIN[r.scenario_name]
    firm = sum(CAPACITY_CREDIT.get(t, 0.9) * r.capacity_gw[t] for t in r.capacity_gw)
    peak_ok = firm >= (r.peak_demand_gw * rm)

    print(f"  {r.scenario_name:<20} {r.status:<10} "
          f"{'✓' if total_gen >= r.annual_demand_twh - 0.01 else '✗':<12} "
          f"{actual_rps:.0%}{'':4} "
          f"{'✓' if peak_ok else '✗'}  (firm={firm:.1f} GW, need={r.peak_demand_gw*rm:.1f} GW)")
print()
print("All constraints satisfied ✓" if all(r.status == "Optimal" for r in results) else "ERRORS detected!")

## 2. Capacity Mix Summary

In [ ]:
rows = []
for r in results:
    for t in TECH_COLORS:
        gw = r.capacity_gw.get(t, 0)
        twh = r.generation_twh.get(t, 0)
        cost = r.cost_breakdown_m.get(t, 0)
        rows.append({"Scenario": r.scenario_name, "Technology": t,
                     "GW": round(gw, 2), "TWh": round(twh, 1),
                     "Cost_M": round(cost, 0)})
df = pd.DataFrame(rows)
pivot = df.pivot_table(values="GW", index="Technology", columns="Scenario")
print(pivot.round(2).to_string())

## 3. Cost Attribution Analysis

In [ ]:
fig = go.Figure()
for tech, color in TECH_COLORS.items():
    fig.add_trace(go.Bar(
        name=tech,
        x=[r.scenario_name for r in results],
        y=[r.cost_breakdown_m.get(tech, 0)/1e3 for r in results],
        marker_color=color,
    ))
fig.update_layout(
    barmode="stack", title="Annualized Cost by Technology ($B/yr)",
    yaxis_title="$B/yr", height=400,
    plot_bgcolor="white", yaxis=dict(gridcolor="#f0f0f0"),
    legend=dict(orientation="h", y=-0.3),
)
fig.show()

## 4. Shadow Price Interpretation

In [ ]:
print("Shadow price interpretation:")
print()
print("The LP dual variables (shadow prices) represent the marginal cost")
print("of relaxing each constraint by one unit.")
print()
print("Key economic insights from binding constraints:")
print()
for r in results:
    total_gen = sum(r.generation_twh.values())
    ren_gen = sum(r.generation_twh[t] for t in r.generation_twh if t in RENEWABLE_TECHS)
    print(f"  {r.scenario_name}:")
    print(f"    System LCOE:        ${r.lcoe_system:.1f}/MWh")
    print(f"    Renewable fraction: {ren_gen/total_gen:.1%}  (target: {r.rps_target:.0%})")
    print(f"    CO₂ intensity:      {r.co2_mt_yr*1e6/total_gen/1e6*1e3:.0f} kg/MWh")
    print(f"    CO₂ reduction:      {r.co2_reduction_pct:.0%} vs 2019 baseline")
    print()

## 5. Marginal Technology Analysis

In [ ]:
atb = opt.atb_df.reset_index()
atb_sorted = atb.sort_values("lcoe_per_mwh")[["technology","lcoe_per_mwh","capacity_factor","type"]]
print("Technologies sorted by LCOE (dispatch merit order):")
print(atb_sorted.to_string(index=False))
print()
print("Note: In the LP, the optimizer selects technologies in merit order")
print("subject to capacity constraints and reliability requirements.")